# Aprendizaje Semi Supervisado

In [ ]:
                         
import sys
assert sys.version_info >= (3, 5)

                                
import sklearn
assert sklearn.__version__ >= "0.20"

                
import numpy as np
import os

                                                   
np.random.seed(42)

                        
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

                           
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "unsupervised_learning"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

## Utilizando Clustering para Aprendizaje Semi-Supervisado

Otro uso de agrupamiento es en el aprendizaje semi-supervisado, en donde solo se cuentan con pocas muestras etiquetadas. El resto de la data esta sin etiqueta

Utilizaremos el _digits dataset_ que es un dataset simple con 1,797 imagenes en escala de grises de 8×8 con digitos de 0 a 9.

In [ ]:
                            
from sklearn.datasets import load_digits

In [ ]:
X_digits, y_digits = load_digits(return_X_y=True)

Se realiza la separacion a datos de entrenamiento y datos de prueba

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_digits, y_digits, random_state=42)

Utilizaremos el modelo de regresion logistica para realizar una prediccion:

In [ ]:
                     
from sklearn.linear_model import LogisticRegression

In [ ]:
log_reg = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
log_reg.fit(X_train, y_train)

In [ ]:
log_reg_score = log_reg.score(X_test, y_test)
log_reg_score

Revisemos ahora el rendimiento de la regresion cuando solo se cuenta con 50 instancias etiquetadas:

In [ ]:
                                              
n_labeled = 50

In [ ]:
log_reg = LogisticRegression(max_iter=5000, solver="lbfgs", random_state=42)
log_reg.fit(X_train[:n_labeled], y_train[:n_labeled])
log_reg.score(X_test, y_test)

Utilizaremos a continuacion las tecnicas de agrupamiento para crear 50 grupos y para cada grupo se buscará una imagen que mas se acerque a su centroide o muestra representativa.

In [ ]:
            
from sklearn.cluster import KMeans

In [ ]:
k = 50

In [ ]:
kmeans = KMeans(n_clusters=k, random_state=42)
X_digits_dist = kmeans.fit_transform(X_train)
representative_digit_idx = np.argmin(X_digits_dist, axis=0)
X_representative_digits = X_train[representative_digit_idx]

Revisamos y etiquetamos manualmente dichas imagenes:

In [ ]:
                                                                         
plt.figure(figsize=(8, 2))
for index, X_representative_digit in enumerate(X_representative_digits):
    plt.subplot(k // 10, 10, index + 1)
    plt.imshow(X_representative_digit.reshape(8, 8), cmap="binary", interpolation="bilinear")
    plt.axis('off')

save_fig("representative_images_diagram", tight_layout=False)
plt.show()

In [ ]:
y_train[representative_digit_idx]

In [ ]:
                                                                            
                                                                                
y_representative_digits = y_train[representative_digit_idx].copy()
print("Etiquetas representativas (5 filas x 10):")
print(y_representative_digits.reshape(5,10))


Revisemos el resultado despues de haber etiquetado manualmente:

In [ ]:
                                      
log_reg = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
log_reg.fit(X_representative_digits, y_representative_digits)
log_reg.score(X_test, y_test)

Tambien es posible copiar las etiquetas al resto de cada grupo.

In [ ]:
                             
y_train_propagated = np.empty(len(X_train), dtype=np.int32)
for i in range(k):
    y_train_propagated[kmeans.labels_==i] = y_representative_digits[i]

In [ ]:
log_reg = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
log_reg.fit(X_train, y_train_propagated)

In [ ]:
log_reg.score(X_test, y_test)

Copiamos la etiqeuta solo al 75% de las muestras mas cercanas al centroide

In [ ]:
                                   
percentile_closest = 75

X_cluster_dist = X_digits_dist[np.arange(len(X_train)), kmeans.labels_]
for i in range(k):
    in_cluster = (kmeans.labels_ == i)
    cluster_dist = X_cluster_dist[in_cluster]
    cutoff_distance = np.percentile(cluster_dist, percentile_closest)
    above_cutoff = (X_cluster_dist > cutoff_distance)
    X_cluster_dist[in_cluster & above_cutoff] = -1

In [ ]:
partially_propagated = (X_cluster_dist != -1)
X_train_partially_propagated = X_train[partially_propagated]
y_train_partially_propagated = y_train_propagated[partially_propagated]

In [ ]:
log_reg = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=42)
log_reg.fit(X_train_partially_propagated, y_train_partially_propagated)

In [ ]:
    
log_reg.score(X_test, y_test)

Veriricando la validez de las etiquetas manuales:

In [ ]:
    
np.mean(y_train_partially_propagated == y_train[partially_propagated])

## Respuestas automáticas de la guía
Se utilizan las etiquetas reales de las imágenes representativas como **referencia de verificación**, en lugar de simular una inspección manual. Ejecuta todas las celdas en orden para obtener las exactitudes de cada etapa.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def entrenar_evaluar(x, etiquetas):
    m=LogisticRegression(max_iter=5000,random_state=42)
    m.fit(x,etiquetas)
    return m.score(X_test,y_test)
print('1. Supervisado completo:',f'{entrenar_evaluar(X_train,y_train):.2%}')
print('2. Solo 50 etiquetas:',f'{entrenar_evaluar(X_train[:50],y_train[:50]):.2%}')
print('4.d. 50 representantes:',f'{entrenar_evaluar(X_representative_digits,y_representative_digits):.2%}')
print('5. Etiquetas propagadas:',f'{entrenar_evaluar(X_train,y_train_propagated):.2%}')
print('6.a. 75% cercano:',f'{entrenar_evaluar(X_train_partially_propagated,y_train_partially_propagated):.2%}')
print('6.b. Calidad de etiquetas propagadas:',f'{np.mean(y_train_partially_propagated == y_train[partially_propagated]):.2%}')


**Conclusión:** El agrupamiento permite seleccionar muestras representativas y propagar sus etiquetas para entrenar con menos ejemplos etiquetados. La propagación puede introducir errores; limitarla al 75% más cercano al centroide busca reducirlos. Compara los valores impresos para determinar si mejoró en este conjunto de datos.